# Weekend 3 — Moduły + organizacja kodu + debug

## Ćwiczenia solved

Ten notebook zawiera rozwiązania krok po kroku do ćwiczeń z organizacji kodu, modułów, clean code, debugowania i type hints.

Pracujemy na historii biznesowej małego projektu: `Retail Order Analyzer`, czyli prostego analizatora zamówień sklepu internetowego.

## Opis datasetu

**Nazwa datasetu:** Online Retail / Online Retail II

**Źródło:** UCI Machine Learning Repository — Online Retail

**Link:** https://archive.ics.uci.edu/dataset/352/online+retail

**Domena biznesowa:** e-commerce, sprzedaż detaliczna online, zamówienia klientów.

**Opis:** Dane przedstawiają transakcje sklepu internetowego. Każdy wiersz to pozycja faktury lub zamówienia. Na podstawie danych ćwiczymy walidację, obliczanie wartości pozycji, raportowanie, organizację kodu i debugowanie.

W repozytorium kursowym używamy pliku `data/online_retail_sample.csv`, czyli mniejszej próbki datasetu, żeby ćwiczenia działały szybko.

### Kolumny

| Kolumna | Typ | Opis biznesowy |
|---|---:|---|
| `invoice_no` | `str` | Numer faktury lub zamówienia. Może wskazywać sprzedaż albo zwrot. |
| `stock_code` | `str` | Kod produktu w magazynie. |
| `description` | `str` | Nazwa lub opis produktu. |
| `quantity` | `int` | Liczba sztuk produktu. Wartości ujemne mogą oznaczać zwrot. |
| `invoice_date` | `str` / `datetime` | Data i godzina wystawienia faktury. |
| `unit_price` | `float` | Cena jednostkowa produktu. |
| `customer_id` | `str` / `int` / `None` | Identyfikator klienta. Może być pusty. |
| `country` | `str` | Kraj klienta lub dostawy. |

### Kategorie biznesowe

W tym bloku nie trenujemy modelu ML, więc nie mamy klasycznej etykiety predykcyjnej. Pracujemy jednak z kategoriami biznesowymi: kraj, produkt, klient, zamówienie, sprzedaż, zwrot, poprawny rekord i niepoprawny rekord.

In [ ]:
from pathlib import Path
from typing import Any
import subprocess
import sys

DATA_PATH = Path("data/online_retail_sample.csv")

print("Data path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

# W tym notebooku używamy małej próbki w pamięci, więc brak pliku CSV nie blokuje ćwiczeń.

# Część 1 — Chaotyczny kod startowy

Zaczynamy od kodu, który działa, ale jest trudny do utrzymania. Potem stopniowo wydzielamy odpowiedzialności.

In [1]:
orders_raw = [
    {"invoice_no": "536365", "stock_code": "85123A", "description": "WHITE HANGING HEART T-LIGHT HOLDER", "quantity": "6", "unit_price": "2.55", "customer_id": "17850", "country": "United Kingdom"},
    {"invoice_no": "536366", "stock_code": "71053", "description": "WHITE METAL LANTERN", "quantity": "-1", "unit_price": "3.39", "customer_id": "17850", "country": "United Kingdom"},
    {"invoice_no": "536367", "stock_code": "84406B", "description": "CREAM CUPID HEARTS COAT HANGER", "quantity": "8", "unit_price": "0", "customer_id": "17850", "country": "France"},
    {"invoice_no": "536368", "stock_code": "84029G", "description": "KNITTED UNION FLAG HOT WATER BOTTLE", "quantity": "3", "unit_price": "3.39", "customer_id": None, "country": "Germany"},
    {"invoice_no": "536369", "stock_code": "22752", "description": "SET 7 BABUSHKA NESTING BOXES", "quantity": "2", "unit_price": "7.65", "customer_id": "13047", "country": "United Kingdom"}
]

s = 0
c = 0
bad = 0

for x in orders_raw:
    q = int(x["quantity"])
    p = float(x["unit_price"])
    if q > 0 and p > 0:
        s = s + q * p
        c = c + 1
    else:
        bad = bad + 1

print("TOTAL", s)
print("VALID", c)
print("Errors", bad)

TOTAL 40.769999999999996
VALID 3
Errors 2


## Zadanie 1 — Mapa odpowiedzialności skryptu

**Rozwiązanie:**

- Dane wejściowe: lista `orders_raw`.
- Konwersja typów: `int(x["quantity"])` oraz `float(x["unit_price"])`.
- Walidacja: warunek `q > 0 and p > 0`.
- Obliczenia: `s = s + q * p`.
- Raportowanie: trzy wywołania `print` na końcu.
- Nieczytelne nazwy: `s`, `c`, `bad`, `x`, `q`, `p`.

Wniosek: skrypt miesza kilka odpowiedzialności w jednym fragmencie. To dobry kandydat do refaktoru.

## Zadanie 2 — Pierwsza funkcja pomocnicza

**Rozwiązanie:** wydzielamy obliczanie wartości jednej pozycji zamówienia do funkcji `calculate_line_total`.

In [ ]:
def calculate_line_total(quantity, unit_price):
    return quantity * unit_price


line_total = calculate_line_total(3, 10.0)
print(line_total)

## Zadanie 3 — Walidacja ilości

**Rozwiązanie:** poprawna ilość powinna być większa od zera.

In [ ]:
def is_valid_quantity(quantity):
    return quantity > 0


print(is_valid_quantity(5))
print(is_valid_quantity(0))
print(is_valid_quantity(-2))

## Zadanie 4 — Walidacja ceny

**Rozwiązanie:** poprawna cena powinna być większa od zera.

In [ ]:
def is_valid_price(unit_price):
    return unit_price > 0


print(is_valid_price(12.5))
print(is_valid_price(0))
print(is_valid_price(-1.0))

## Zadanie 5 — Walidacja całego zamówienia

**Rozwiązanie:** funkcja konwertuje dane z tekstu na liczby i używa dwóch małych walidatorów.

In [ ]:
def is_valid_order(order):
    quantity = int(order["quantity"])
    unit_price = float(order["unit_price"])

    return is_valid_quantity(quantity) and is_valid_price(unit_price)


for order in orders_raw:
    print(order["invoice_no"], is_valid_order(order))

## Zadanie 6 — Czytelne nazwy zmiennych

**Rozwiązanie:** zastępujemy skróty nazwami biznesowymi.

In [ ]:
total_revenue = 0.0
valid_orders_count = 0
invalid_orders_count = 0

for order in orders_raw:
    quantity = int(order["quantity"])
    unit_price = float(order["unit_price"])

    if is_valid_quantity(quantity) and is_valid_price(unit_price):
        total_revenue += calculate_line_total(quantity, unit_price)
        valid_orders_count += 1
    else:
        invalid_orders_count += 1

print("Total revenue:", total_revenue)
print("Valid orders:", valid_orders_count)
print("Invalid orders:", invalid_orders_count)

## Zadanie 7 — Funkcja `calculate_total_revenue`

**Rozwiązanie:** funkcja liczy przychód i zwraca wynik. Nie drukuje raportu, bo to będzie osobna odpowiedzialność.

In [ ]:
def calculate_total_revenue(orders):
    total_revenue = 0.0

    for order in orders:
        if is_valid_order(order):
            quantity = int(order["quantity"])
            unit_price = float(order["unit_price"])
            total_revenue += calculate_line_total(quantity, unit_price)

    return total_revenue


print(calculate_total_revenue(orders_raw))

## Zadanie 8 — Funkcja raportująca

**Rozwiązanie:** funkcja raportująca tylko prezentuje wynik. Nie liczy przychodu od nowa.

In [ ]:
def print_revenue_report(total_revenue):
    print("=" * 30)
    print("Retail revenue report")
    print(f"Total revenue: {total_revenue:.2f}")
    print("=" * 30)


total_revenue = calculate_total_revenue(orders_raw)
print_revenue_report(total_revenue)

# Część 2 — Moduły i struktura projektu

Teraz przenosimy funkcje do osobnych plików. W notebooku zapisujemy pliki `.py` za pomocą `Path.write_text()`.

## Zadanie 9 — Utworzenie katalogu projektu

**Rozwiązanie:** tworzymy katalog `retail_order_analyzer`.

In [ ]:
PROJECT_DIR = Path("retail_order_analyzer")
PROJECT_DIR.mkdir(exist_ok=True)
(PROJECT_DIR / "__init__.py").write_text("", encoding="utf-8")

print(PROJECT_DIR.exists())
print((PROJECT_DIR / "__init__.py").exists())

## Zadanie 10 — Moduł `validators.py`

**Rozwiązanie:** walidacja ilości, ceny i całego zamówienia trafia do jednego modułu.

In [ ]:
validators_code = '''
MIN_VALID_QUANTITY = 1
MIN_VALID_PRICE = 0.0


def is_valid_quantity(quantity):
    return quantity >= MIN_VALID_QUANTITY


def is_valid_price(unit_price):
    return unit_price > MIN_VALID_PRICE


def is_valid_order(order):
    quantity = int(order["quantity"])
    unit_price = float(order["unit_price"])

    return is_valid_quantity(quantity) and is_valid_price(unit_price)
'''

(PROJECT_DIR / "validators.py").write_text(validators_code.strip() + "\n", encoding="utf-8")

print((PROJECT_DIR / "validators.py").read_text(encoding="utf-8"))

## Zadanie 11 — Moduł `pricing.py`

**Rozwiązanie:** logika liczenia trafia do `pricing.py`. Moduł importuje tylko to, czego potrzebuje.

In [ ]:
pricing_code = '''
try:
    from .validators import is_valid_order
except ImportError:
    from validators import is_valid_order


def calculate_line_total(quantity, unit_price):
    return quantity * unit_price


def calculate_total_revenue(orders):
    total_revenue = 0.0

    for order in orders:
        if is_valid_order(order):
            quantity = int(order["quantity"])
            unit_price = float(order["unit_price"])
            total_revenue += calculate_line_total(quantity, unit_price)

    return total_revenue
'''

(PROJECT_DIR / "pricing.py").write_text(pricing_code.strip() + "\n", encoding="utf-8")

print((PROJECT_DIR / "pricing.py").read_text(encoding="utf-8"))

## Zadanie 12 — Moduł `reports.py`

**Rozwiązanie:** raportowanie trafia do osobnego modułu.

In [ ]:
reports_code = '''
def print_revenue_report(total_revenue):
    print("=" * 30)
    print("Retail revenue report")
    print(f"Total revenue: {total_revenue:.2f}")
    print("=" * 30)
'''

(PROJECT_DIR / "reports.py").write_text(reports_code.strip() + "\n", encoding="utf-8")

print((PROJECT_DIR / "reports.py").read_text(encoding="utf-8"))

## Zadanie 13 — Plik `main.py`

**Rozwiązanie:** `main.py` ma być krótkim punktem wejścia. Szczegóły są w modułach.

In [ ]:
main_code = '''
from pricing import calculate_total_revenue
from reports import print_revenue_report


orders = [
    {"invoice_no": "536365", "quantity": "6", "unit_price": "2.55", "country": "United Kingdom"},
    {"invoice_no": "536366", "quantity": "-1", "unit_price": "3.39", "country": "United Kingdom"},
    {"invoice_no": "536367", "quantity": "8", "unit_price": "0", "country": "France"},
    {"invoice_no": "536368", "quantity": "3", "unit_price": "3.39", "country": "Germany"},
]


def main():
    total_revenue = calculate_total_revenue(orders)
    print_revenue_report(total_revenue)


if __name__ == "__main__":
    main()
'''

(PROJECT_DIR / "main.py").write_text(main_code.strip() + "\n", encoding="utf-8")

print((PROJECT_DIR / "main.py").read_text(encoding="utf-8"))

## Zadanie 14 — Uruchomienie projektu z terminala

**Rozwiązanie:** uruchamiamy `main.py` jako skrypt. W notebooku używamy `subprocess.run`, żeby zobaczyć wynik.

In [ ]:
result = subprocess.run(
    [sys.executable, str(PROJECT_DIR / "main.py")],
    capture_output=True,
    text=True,
    check=False,
)

print("Return code:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

## Zadanie 15 — Review importów

**Rozwiązanie:**

- W projekcie nie ma `import *`.
- `main.py` importuje jawnie: `calculate_total_revenue` i `print_revenue_report`.
- `pricing.py` importuje tylko `is_valid_order`.
- `validators.py` nie importuje `pricing.py`, więc nie tworzymy importu cyklicznego.

To jest prosta i czytelna struktura zależności: `main.py` używa modułów, `pricing.py` używa walidatora, a raportowanie jest oddzielone od obliczeń.

# Część 3 — Clean code i refaktor

W tej części poprawiamy czytelność: stałe, nazwy, funkcje pomocnicze i AI review.

## Zadanie 16 — Stałe zamiast magic numbers

**Rozwiązanie:** reguły biznesowe zapisujemy jako nazwane stałe.

In [ ]:
MIN_VALID_QUANTITY = 1
MIN_VALID_PRICE = 0.0
UNKNOWN_CUSTOMER_ID = "UNKNOWN"


def clean_is_valid_quantity(quantity):
    return quantity >= MIN_VALID_QUANTITY


def clean_is_valid_price(unit_price):
    return unit_price > MIN_VALID_PRICE


print(clean_is_valid_quantity(1))
print(clean_is_valid_price(0.01))
print(UNKNOWN_CUSTOMER_ID)

## Zadanie 17 — Funkcja `is_return`

**Rozwiązanie:** zwrot rozpoznajemy po ujemnej ilości.

In [ ]:
def is_return(quantity):
    return quantity < 0


print(is_return(-1))
print(is_return(0))
print(is_return(5))

## Zadanie 18 — Normalizacja kraju

**Rozwiązanie:** usuwamy nadmiarowe spacje i ujednolicamy wielkość liter. Dla pustego kraju zwracamy `Unknown`.

In [ ]:
DEFAULT_COUNTRY = "Unknown"


def normalize_country_name(country):
    normalized_country = country.strip()

    if normalized_country == "":
        return DEFAULT_COUNTRY

    return normalized_country.title()


print(normalize_country_name(" united kingdom "))
print(normalize_country_name("GERMANY"))
print(normalize_country_name(""))

## Zadanie 19 — AI review nazw

**Przykładowy dobry prompt:**

Sprawdź poniższy kod jako reviewer Pythona dla początkującego juniora. Skup się tylko na nazwach funkcji, nazwach zmiennych i odpowiedzialnościach. Nie zmieniaj logiki biznesowej. Zaproponuj maksymalnie 5 konkretnych poprawek i krótko wyjaśnij każdą.

**Przykładowe decyzje po review:**

- Przyjmuję zmianę `s` na `total_revenue`, bo nazwa mówi, co przechowuje zmienna.
- Przyjmuję zmianę `x` na `order`, bo iterujemy po zamówieniach.
- Przyjmuję wydzielenie `calculate_line_total`, bo usuwa duplikację i nadaje nazwę obliczeniu.
- Odrzucam sugestię wprowadzania klas, bo na tym etapie wystarczą funkcje i moduły.

# Część 4 — Debugowanie

W rozwiązaniach pokazujemy diagnozę bez celowego zatrzymywania całego notebooka wyjątkiem. W prawdziwym edytorze warto uruchomić te przykłady w debuggerze i ustawiać breakpointy.

## Zadanie 20 — Czytanie tracebacka

**Rozwiązanie:** uruchamiamy błąd w kontrolowany sposób przez `try/except`, żeby notebook działał dalej. W normalnym debugowaniu najpierw czytamy ostatnią linię tracebacka.

In [ ]:
def broken_line_total(quantity, unit_price):
    return quantity * unit_price


quantity_from_csv = "5"
unit_price_from_csv = 12.5

try:
    broken_line_total(quantity_from_csv, unit_price_from_csv)
except TypeError as error:
    print("Typ błędu:", type(error).__name__)
    print("Komunikat:", error)
    print("Podejrzana wartość:", quantity_from_csv)
    print("Typ podejrzanej wartości:", type(quantity_from_csv).__name__)
    print("Hipoteza: quantity zostało wczytane jako tekst, a funkcja oczekiwała liczby.")

## Zadanie 21 — Pierwszy breakpoint

**Rozwiązanie koncepcyjne:** ustaw breakpoint na linii `return quantity * unit_price`, a potem sprawdź wartości i typy argumentów.

W notebooku możemy zasymulować obserwację przez wypisanie wartości i typów.

In [ ]:
def debug_line_total(quantity, unit_price):
    print("DEBUG quantity:", quantity, type(quantity).__name__)
    print("DEBUG unit_price:", unit_price, type(unit_price).__name__)
    return int(quantity) * unit_price


print(debug_line_total("4", 9.99))

## Zadanie 22 — `step into`

**Rozwiązanie:** błąd pojawia się w `parse_quantity`, bo tekst `not-a-number` nie może zostać zamieniony na `int`.

In [ ]:
def parse_quantity(raw_quantity):
    return int(raw_quantity)


def process_debug_order(order):
    quantity = parse_quantity(order["quantity"])
    return quantity


debug_order = {"quantity": "not-a-number"}

try:
    process_debug_order(debug_order)
except ValueError as error:
    print("Typ błędu:", type(error).__name__)
    print("Komunikat:", error)
    print("Przyczyna: parse_quantity dostało tekst, który nie jest liczbą.")
    print("Poprawka: dodać walidację danych wejściowych albo obsługę błędu konwersji.")

## Zadanie 23 — Błąd `None`

**Rozwiązanie:** jeśli `customer_id` może być `None`, obsługujemy ten przypadek przed `.strip()`.

In [ ]:
def normalize_customer_id(customer_id):
    if customer_id is None:
        return UNKNOWN_CUSTOMER_ID

    return customer_id.strip()


print(normalize_customer_id(" 17850 "))
print(normalize_customer_id(None))

## Zadanie 24 — Błąd logiczny w warunku

**Rozwiązanie:** zwrot to ilość ujemna, więc warunek powinien być `quantity < 0`, a nie `quantity > 0`.

In [ ]:
def fixed_is_return(quantity):
    return quantity < 0


print("-2:", fixed_is_return(-2))
print("0:", fixed_is_return(0))
print("3:", fixed_is_return(3))

## Zadanie 25 — Call stack

**Rozwiązanie:** najniższa funkcja dostaje `quantity` jako tekst. Call stack pomaga odkryć, że wartość przyszła z `high_level_report`.

In [ ]:
def low_level_calculation(quantity, unit_price):
    print("low_level_calculation quantity:", quantity, type(quantity).__name__)
    print("low_level_calculation unit_price:", unit_price, type(unit_price).__name__)
    return int(quantity) * unit_price


def middle_level_processing(order):
    return low_level_calculation(order["quantity"], order["unit_price"])


def high_level_report():
    order = {"quantity": "3", "unit_price": 10.0}
    return middle_level_processing(order)


print(high_level_report())
print("W debuggerze call stack wyglądałby logicznie: high_level_report -> middle_level_processing -> low_level_calculation.")

## Zadanie 26 — AI jako asystent debugowania

**Przykładowy prompt:**

Mam błąd w Pythonie. Oczekuję, że funkcja policzy wartość pozycji zamówienia jako `quantity * unit_price`. Traceback kończy się komunikatem: `TypeError: can't multiply sequence by non-int of type 'float'`. Minimalny kod: `quantity = "5"`, `unit_price = 12.5`, `return quantity * unit_price`. Podaj 3 hipotezy przyczyny i napisz, jak sprawdzić każdą w debuggerze. Nie przepisuj całego projektu.

**Przykładowe hipotezy:**

- `quantity` jest tekstem zamiast liczbą. Sprawdzić typ w debuggerze.
- Konwersja danych z CSV nie została wykonana. Sprawdzić miejsce wczytywania danych.
- Funkcja ma niejasny kontrakt. Dodać type hints i porównać je z realnymi wartościami.

**Decyzja:** najpierw sprawdzamy typ `quantity`, potem cofamy się do miejsca wczytywania danych.

# Część 5 — Type hints

Type hints opisują kontrakt funkcji. Teraz dodajemy typy do najważniejszych funkcji.

## Zadanie 27 — Type hints dla funkcji obliczeniowych

**Rozwiązanie:** `quantity` to `int`, `unit_price` to `float`, wynik to `float`.

In [ ]:
Order = dict[str, str | int | float | None]


def typed_calculate_line_total(quantity: int, unit_price: float) -> float:
    return quantity * unit_price


def typed_calculate_total_revenue(orders: list[Order]) -> float:
    total_revenue = 0.0

    for order in orders:
        quantity = int(order["quantity"])
        unit_price = float(order["unit_price"])

        if quantity > 0 and unit_price > 0:
            total_revenue += typed_calculate_line_total(quantity, unit_price)

    return total_revenue


print(typed_calculate_line_total(2, 12.5))
print(typed_calculate_total_revenue(orders_raw))

## Zadanie 28 — Type hints dla walidatorów

**Rozwiązanie:** funkcje z `is_` zwracają `bool`.

In [ ]:
def typed_is_valid_quantity(quantity: int) -> bool:
    return quantity > 0


def typed_is_valid_price(unit_price: float) -> bool:
    return unit_price > 0


def typed_is_return(quantity: int) -> bool:
    return quantity < 0


print(typed_is_valid_quantity(10))
print(typed_is_valid_price(2.5))
print(typed_is_return(-1))

## Zadanie 29 — `str | None` w praktyce

**Rozwiązanie:** typ mówi, że argument może być tekstem albo `None`. Kod obsługuje oba przypadki.

In [ ]:
def safe_normalize_customer_id(customer_id: str | None) -> str:
    if customer_id is None:
        return UNKNOWN_CUSTOMER_ID

    return customer_id.strip()


print(safe_normalize_customer_id(" 17850 "))
print(safe_normalize_customer_id(None))

## Zadanie 30 — Typowanie słownika `Order`

**Rozwiązanie:** tworzymy alias typu, żeby sygnatury funkcji były czytelniejsze.

In [ ]:
Order = dict[str, str | int | float | None]


def get_invoice_no(order: Order) -> str:
    return str(order["invoice_no"])


sample_order: Order = {
    "invoice_no": "536365",
    "quantity": 6,
    "unit_price": 2.55,
    "customer_id": None,
}

print(get_invoice_no(sample_order))

## Zadanie 31 — Typowanie listy zamówień

**Rozwiązanie:** funkcje raportowe zwykle przyjmują `list[Order]`.

In [ ]:
def count_orders(orders: list[Order]) -> int:
    return len(orders)


def count_orders_by_country(orders: list[Order], country: str) -> int:
    count = 0

    for order in orders:
        order_country = str(order["country"])

        if normalize_country_name(order_country) == normalize_country_name(country):
            count += 1

    return count


print(count_orders(orders_raw))
print(count_orders_by_country(orders_raw, "United Kingdom"))

## Zadanie 32 — Debug + typy razem

**Rozwiązanie:** type hints mówią, że `quantity` powinno być `int`, ale realna wartość może być tekstem. W debuggerze porównujemy kontrakt z runtime.

In [ ]:
def calculate_discounted_total(quantity: int, unit_price: float, discount_rate: float) -> float:
    return quantity * unit_price * (1 - discount_rate)


debug_quantity = "10"
debug_unit_price = 5.0
debug_discount_rate = 0.1

print("Type hint dla quantity: int")
print("Realny typ debug_quantity:", type(debug_quantity).__name__)

fixed_quantity = int(debug_quantity)
print(calculate_discounted_total(fixed_quantity, debug_unit_price, debug_discount_rate))

# Część 6 — Mini-projekt kończący weekend

Na końcu składamy finalną wersję projektu z modułami, type hints i krótką notatką z debugowania.

## Zadanie 33 — Mini-projekt: finalna struktura

**Rozwiązanie:** zapisujemy finalne wersje modułów. Kod jest prosty, ale ma jasny podział odpowiedzialności.

In [ ]:
final_validators_code = '''
Order = dict[str, str | int | float | None]

MIN_VALID_QUANTITY = 1
MIN_VALID_PRICE = 0.0
UNKNOWN_CUSTOMER_ID = "UNKNOWN"
DEFAULT_COUNTRY = "Unknown"


def is_valid_quantity(quantity: int) -> bool:
    return quantity >= MIN_VALID_QUANTITY


def is_valid_price(unit_price: float) -> bool:
    return unit_price > MIN_VALID_PRICE


def is_return(quantity: int) -> bool:
    return quantity < 0


def normalize_customer_id(customer_id: str | None) -> str:
    if customer_id is None:
        return UNKNOWN_CUSTOMER_ID

    return customer_id.strip()


def normalize_country_name(country: str) -> str:
    normalized_country = country.strip()

    if normalized_country == "":
        return DEFAULT_COUNTRY

    return normalized_country.title()


def is_valid_order(order: Order) -> bool:
    quantity = int(order["quantity"])
    unit_price = float(order["unit_price"])

    return is_valid_quantity(quantity) and is_valid_price(unit_price)
'''

final_pricing_code = '''
try:
    from .validators import Order, is_valid_order
except ImportError:
    from validators import Order, is_valid_order


def calculate_line_total(quantity: int, unit_price: float) -> float:
    return quantity * unit_price


def calculate_total_revenue(orders: list[Order]) -> float:
    total_revenue = 0.0

    for order in orders:
        if is_valid_order(order):
            quantity = int(order["quantity"])
            unit_price = float(order["unit_price"])
            total_revenue += calculate_line_total(quantity, unit_price)

    return total_revenue


def count_valid_orders(orders: list[Order]) -> int:
    valid_orders_count = 0

    for order in orders:
        if is_valid_order(order):
            valid_orders_count += 1

    return valid_orders_count
'''

final_reports_code = '''
def print_revenue_report(total_revenue: float, valid_orders_count: int) -> None:
    print("=" * 30)
    print("Retail revenue report")
    print(f"Total revenue: {total_revenue:.2f}")
    print(f"Valid orders: {valid_orders_count}")
    print("=" * 30)
'''

final_debug_examples_code = '''
def example_type_error() -> None:
    quantity = "5"
    unit_price = 12.5

    print("Przed poprawką quantity ma typ:", type(quantity).__name__)
    fixed_quantity = int(quantity)
    print("Po poprawce wynik:", fixed_quantity * unit_price)


def example_none_error() -> None:
    customer_id = None

    if customer_id is None:
        print("Customer ID is missing")
    else:
        print(customer_id.strip())
'''

final_main_code = '''
from pricing import calculate_total_revenue, count_valid_orders
from reports import print_revenue_report


orders = [
    {"invoice_no": "536365", "quantity": "6", "unit_price": "2.55", "customer_id": "17850", "country": "United Kingdom"},
    {"invoice_no": "536366", "quantity": "-1", "unit_price": "3.39", "customer_id": "17850", "country": "United Kingdom"},
    {"invoice_no": "536367", "quantity": "8", "unit_price": "0", "customer_id": "17850", "country": "France"},
    {"invoice_no": "536368", "quantity": "3", "unit_price": "3.39", "customer_id": None, "country": "Germany"},
]


def main() -> None:
    total_revenue = calculate_total_revenue(orders)
    valid_orders_count = count_valid_orders(orders)
    print_revenue_report(total_revenue, valid_orders_count)


if __name__ == "__main__":
    main()
'''

(PROJECT_DIR / "validators.py").write_text(final_validators_code.strip() + "\n", encoding="utf-8")
(PROJECT_DIR / "pricing.py").write_text(final_pricing_code.strip() + "\n", encoding="utf-8")
(PROJECT_DIR / "reports.py").write_text(final_reports_code.strip() + "\n", encoding="utf-8")
(PROJECT_DIR / "debug_examples.py").write_text(final_debug_examples_code.strip() + "\n", encoding="utf-8")
(PROJECT_DIR / "main.py").write_text(final_main_code.strip() + "\n", encoding="utf-8")

for path in sorted(PROJECT_DIR.glob("*.py")):
    print(path)

In [ ]:
result = subprocess.run(
    [sys.executable, str(PROJECT_DIR / "main.py")],
    capture_output=True,
    text=True,
    check=False,
)

print("Return code:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

## Zadanie 34 — Notatka techniczna z debugowania

**Rozwiązanie:** zapisujemy krótką notatkę w `debug_notes.md`.

In [ ]:
debug_notes = '''
# Debug note

## Objaw

Program wyświetlał błąd typu podczas liczenia wartości pozycji zamówienia.

## Przyczyna

Pole `quantity` zostało wczytane jako tekst, a funkcja licząca wartość pozycji oczekiwała liczby całkowitej.

## Jak to sprawdziłem/sprawdziłam w debuggerze

Ustawiłem breakpoint w funkcji liczącej wartość pozycji i sprawdziłem wartości oraz typy argumentów. `quantity` miało wartość `"5"` i typ `str`, mimo że type hint wskazywał `int`.

## Poprawka

Dodałem konwersję `int(order["quantity"])` przed wywołaniem funkcji obliczeniowej oraz type hints w funkcjach obliczeniowych.
'''

(PROJECT_DIR / "debug_notes.md").write_text(debug_notes.strip() + "\n", encoding="utf-8")

print((PROJECT_DIR / "debug_notes.md").read_text(encoding="utf-8"))

# Checklista końcowa — rozwiązana

- [x] Projekt ma `main.py` jako punkt wejścia.
- [x] Walidacja jest w `validators.py`.
- [x] Obliczenia są w `pricing.py`.
- [x] Raportowanie jest w `reports.py`.
- [x] Przykłady debugowania są w `debug_examples.py`.
- [x] Powstał plik `debug_notes.md`.
- [x] Najważniejsze funkcje mają type hints.
- [x] Obsłużono `None` w `normalize_customer_id`.
- [x] Kod można uruchomić jednym poleceniem.

Najważniejszy wniosek: podział kodu na moduły, dobre nazwy, debugowanie i type hints wzajemnie się wzmacniają. Im czytelniejszy kod, tym łatwiej znaleźć błąd. Im lepsze typy, tym szybciej widać, gdzie rzeczywiste dane nie zgadzają się z kontraktem funkcji.